# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [11]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [4]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.

# TODO: Add documents for the AI agent to search.

DOCUMENTS = {
    "artificial_intelligence": (
        "Artificial Intelligence enables computers to perform tasks that normally require human intelligence. "
        "It is widely used in healthcare, education, finance, and transportation. "
        "Machine learning and deep learning are important technologies used in AI systems. "
        "AI assistants can understand user queries and generate meaningful responses."
    ),

    "renewable_energy": (
        "Renewable energy is obtained from naturally replenished sources. "
        "Solar energy uses sunlight to generate electricity through solar panels. "
        "Wind energy uses turbines to convert wind movement into electrical power. "
        "Renewable energy helps reduce dependence on fossil fuels and environmental pollution."
    ),

    "space_exploration": (
        "Space exploration involves studying celestial objects and outer space. "
        "Organizations such as NASA and ISRO conduct space missions and scientific research. "
        "Satellites are used for communication, navigation, weather forecasting, and observation. "
        "Modern space technology is helping scientists explore the Moon, Mars, and beyond."
    ),

    "cybersecurity": (
        "Cybersecurity focuses on protecting computers, networks, and digital information. "
        "Strong passwords and multi-factor authentication help protect online accounts. "
        "Phishing attacks trick users into revealing confidential information. "
        "Regular software updates and security awareness help reduce cyber threats."
    ),

    "python_programming": (
        "Python is a high-level programming language known for its simple syntax. "
        "It is commonly used in artificial intelligence, data science, automation, and web development. "
        "Python supports programming concepts such as functions, loops, classes, and modules. "
        "Popular Python libraries include NumPy, Pandas, TensorFlow, and Matplotlib."
    )
}


In [6]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("artificial intelligence"))

[Source: python_programming]
Python is a high-level programming language known for its simple syntax. It is commonly used in artificial intelligence, data science, automation, and web development. Python supports programming concepts such as functions, loops, classes, and modules. Popular Python libraries include NumPy, Pandas, TensorFlow, and Matplotlib.

[Source: artificial_intelligence]
Artificial Intelligence enables computers to perform tasks that normally require human intelligence. It is widely used in healthcare, education, finance, and transportation. Machine learning and deep learning are important technologies used in AI systems. AI assistants can understand user queries and generate meaningful responses.


In [7]:
# @title Ask gemini to search for something

import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.8-flash",
    tools=[search_documents],
    system_instruction=(
        """
        You are an intelligent research assistant.

        Your task is to answer user questions using the
        available document collection.

        When you need specific information, use the
        search_documents tool.

        Only answer based on the information found
        in the documents.

        If the documents do not contain the answer,
        clearly state that the information is unavailable.

        Provide clear, accurate and structured answers.
        """
    )
)

print("Model ready:", model.model_name)


/tmp/ipykernel_33765/3800814335.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Model ready: models/gemini-3.8-flash


In [12]:

!pip install -q google-genai
import time
from google import genai
from google.genai import types

client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"]
)

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the document collection for paragraphs relevant to the query.",
        parameters={
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
)

def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model="gemini-2.5-flash",
                contents=contents,
                config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        contents.append(model_content)

        calls = [
            p.function_call
            for p in model_content.parts
            if p.function_call
        ]

        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        result_parts = []

        for call in calls:
            args = dict(call.args)

            print(f"Step {step}: model calls {call.name}({args})")

            result = search_documents(**args)

            print(f"Result preview: {str(result)[:100]}...")

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        contents.append(
            types.Content(role="user", parts=result_parts)
        )

    return "(Stopped: hit max_steps)"


_ = run_agent(
    "What topics are students supposed to share documents on?"
)


Question: What topics are students supposed to share documents on?
--------------------------------------------------


ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'The Gemini API free tier is not available for this project. To use the Gemini API, please set up billing to upgrade to the paid tier at https://aistudio.google.com/billing. For more information, see https://ai.google.dev/gemini-api/docs/troubleshooting#project-unavailable.', 'status': 'PERMISSION_DENIED'}}

In [14]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "What topics are students supposed to share documents on?",
    "How can students collaborate on group projects?",
    "What resources are available for students?",
    "How can students ask questions and get help?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)

    print("\n" + "=" * 60)
    run_agent(q)






Question: What topics are students supposed to share documents on?
--------------------------------------------------


ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'The Gemini API free tier is not available for this project. To use the Gemini API, please set up billing to upgrade to the paid tier at https://aistudio.google.com/billing. For more information, see https://ai.google.dev/gemini-api/docs/troubleshooting#project-unavailable.', 'status': 'PERMISSION_DENIED'}}

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)